# Target Construction — `cannot_afford`
# Notebook by DR

Builds the modelling outcome: the share of each country's population that cannot afford a
healthy diet.

The outcome exists in none of the prepared files. It is constructed by setting a poverty
line equal to the country's own cost of a healthy diet divided by the food budget share,
then querying PIP at that line:

```
line = CoHD_PPP / 0.52
GET {PIP_BASE}/pip?country={iso3}&year={y}&povline={line}&ppp_version=2021&fill_gaps=false
```

The returned `headcount` becomes `cannot_afford`.

**`CoHD_headcount` in fpn.csv is not the target.** It is the World Bank's published
version, imputed from regional aggregates where country data is missing. It is retained
only as a comparison: the gap flags which countries rest on estimate rather than
measurement.

**Input**  `data_cache.zip`
**Output** `data_cache/target.csv`

Section 6 makes roughly 500 API calls and takes about five minutes; it
is resumable if the connection drops. The cache reset in section 1 sits after the unzip on
purpose, so a full run always reflects the settings in section 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Load the prepared data

In [ ]:
# Retrieves data_cache.zip. Tries Drive first, then the current session, then
# falls back to a manual upload.
from google.colab import drive, files
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_ZIP = Path('/content/drive/MyDrive/data_cache.zip')
LOCAL_ZIP = Path('data_cache.zip')

if DRIVE_ZIP.exists():
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print('copied data_cache.zip from Drive')
elif LOCAL_ZIP.exists():
    print('data_cache.zip already present in this session')
else:
    print('Not found in Drive. Select data_cache.zip from the local machine:')
    uploaded = files.upload()
    for name in uploaded:
        if name != 'data_cache.zip':
            Path(name).rename('data_cache.zip')
    shutil.copy(LOCAL_ZIP, DRIVE_ZIP)
    print('uploaded and saved a copy to Drive')

Mounted at /content/drive
copied data_cache.zip from Drive


In [ ]:
!unzip -q -o data_cache.zip
!ls -lh data_cache

total 1.8M
-rw-r--r-- 1 root root  17K Sep  9 07:09 countries.csv
-rw-r--r-- 1 root root  21K Sep  9 07:10 coverage_report.csv
-rw-r--r-- 1 root root 340K Sep  9 07:09 fpn.csv
-rw-r--r-- 1 root root 8.3K Sep  9 07:10 import_dependence.csv
-rw-r--r-- 1 root root 169K Sep  9 07:09 pip_baseline.csv
-rw-r--r-- 1 root root  30K Sep 10 00:59 pip_target_raw.csv
-rw-r--r-- 1 root root 4.9K Sep  9 07:09 pip_vintage.csv
-rw-r--r-- 1 root root  65K Sep 10 00:59 target.csv
-rw-r--r-- 1 root root 1.2M Sep  9 07:10 wdi.csv


### Cache reset

The zip carries any `pip_target_raw.csv` from a previous run, and the unzip above restores
it. The reset therefore has to happen here rather than before section 1, or the query loop
resumes from results computed at a different poverty line.

In [ ]:
# Removes cached query results carried in from a previous run
import os

STALE = 'data_cache/pip_target_raw.csv'
if os.path.exists(STALE):
    os.remove(STALE)
    print('removed stale query cache -- all queries will be re-fetched')
else:
    print('no stale cache present')

removed stale query cache -- all queries will be re-fetched


In [ ]:
# Confirm the expected inputs are present before proceeding
from pathlib import Path

EXPECTED = ['countries.csv', 'fpn.csv', 'wdi.csv', 'pip_baseline.csv',
            'pip_vintage.csv', 'import_dependence.csv', 'coverage_report.csv']

CACHE = Path('data_cache')
missing = [f for f in EXPECTED if not (CACHE / f).exists()]

if missing:
    raise FileNotFoundError(f'missing inputs: {missing}')
print('all 7 input files present')

all 7 input files present


## 2. Configuration

`PPP_VERSION = 2021` is required, not a preference. FPN v5.0 is denominated in 2021 PPPs
and a mismatch produces wrong numbers that look reasonable.

`FOOD_BUDGET_SHARE = 0.52` reflects the affordability convention: a diet is unaffordable
when its cost exceeds the portion of income plausibly available for food. The 52% figure
comes from observed food spending in low-income countries (Herforth et al.). Dividing by
it raises the poverty line to the income level at which the diet becomes affordable.

In [ ]:
import time
import warnings
from pathlib import Path

import pandas as pd
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 60)

CACHE = Path('data_cache')
PIP_BASE = 'https://api.worldbank.org/pip/v1'
RAW = CACHE / 'pip_target_raw.csv'

PPP_VERSION = 2021          # required by FPN v5.0, do not vary
FOOD_BUDGET_SHARE = 0.52    # poverty line = CoHD_PPP / 0.52
MIN_YEAR = 2017             # first year FPN publishes the cost of a healthy diet
PAUSE = 0.4                 # inter-request delay

TARGET_NAME = 'cannot_afford'

print(f'PPP basis         : {PPP_VERSION}')
print(f'food budget share : {FOOD_BUDGET_SHARE}')
print(f'poverty line      : CoHD_PPP / {FOOD_BUDGET_SHARE}')
print(f'target column     : {TARGET_NAME}')

PPP basis         : 2021
food budget share : 0.52
poverty line      : CoHD_PPP / 0.52
target column     : cannot_afford


## 3. Define the sample and the country-specific thresholds

In [ ]:
fpn      = pd.read_csv(CACHE / 'fpn.csv')
coverage = pd.read_csv(CACHE / 'coverage_report.csv')
pip_base = pd.read_csv(CACHE / 'pip_baseline.csv')

# The certified sample: present in FPN, PIP and WDI, with a survey under 10 years old
usable = coverage.loc[coverage['usable'], 'iso3'].tolist()
print(f'usable sample: {len(usable)} countries')

cohd = (fpn[(fpn['indicator'] == 'CoHD_PPP') & fpn['iso3'].isin(usable)]
        [['iso3', 'year', 'value']]
        .rename(columns={'value': 'cohd_ppp'}))
cohd = cohd[cohd['year'] >= MIN_YEAR].dropna(subset=['cohd_ppp'])

print(f'CoHD observations: {len(cohd)} country-years, '
      f'{cohd["iso3"].nunique()} countries, {cohd["year"].min()}-{cohd["year"].max()}')

# FPN v5.0 withholds PPP-denominated indicators for six countries. Local currency values
# exist but are not comparable across countries, so these drop out here.
no_ppp = sorted(set(usable) - set(cohd['iso3']))
if no_ppp:
    print(f'no PPP-denominated CoHD ({len(no_ppp)}): {no_ppp}')

usable sample: 138 countries
CoHD observations: 1197 country-years, 133 countries, 2017-2025
no PPP-denominated CoHD (5): ['ARG', 'MMR', 'THA', 'TJK', 'ZWE']


In [ ]:
# Restrict to genuine survey years. In these country-years the income side is measured
# rather than interpolated, so the constructed target is an observation.
surveys = (pip_base[pip_base['iso3'].isin(usable)]
           [['iso3', 'reporting_year']]
           .drop_duplicates()
           .rename(columns={'reporting_year': 'year'}))
surveys['year'] = pd.to_numeric(surveys['year'], errors='coerce').astype('Int64')

grid = cohd.merge(surveys, on=['iso3', 'year'], how='inner')
grid['povline'] = (grid['cohd_ppp'] / FOOD_BUDGET_SHARE).round(4)

print(f'query grid: {len(grid)} country-years')
print(f'countries represented: {grid["iso3"].nunique()} of {len(usable)}')
print()
print('surveys per country:')
print(grid.groupby('iso3').size().value_counts().sort_index().to_string())
print()
print('poverty lines in use (PPP $/day):')
print(grid['povline'].describe().round(2).to_string())

query grid: 507 country-years
countries represented: 129 of 138

surveys per country:
1    43
2    19
3     6
4     5
5     7
6     6
7    30
8    10
9     3

poverty lines in use (PPP $/day):
count    507.00
mean       6.58
std        1.55
min        3.27
25%        5.48
50%        6.40
75%        7.42
max       11.87


## 4. Smoke test

One query is inspected before the full run, to confirm the parameters are accepted and the
response carries the expected fields.

In [ ]:
test = grid.iloc[0]
print(f'testing {test["iso3"]} {test["year"]}')
print(f'  diet cost   : {test["cohd_ppp"]:.2f} PPP $/day')
print(f'  poverty line: {test["povline"]:.2f} PPP $/day  (cost / {FOOD_BUDGET_SHARE})')
print()

r = requests.get(f'{PIP_BASE}/pip', params={
    'country': test['iso3'],
    'year': int(test['year']),
    'povline': float(test['povline']),
    'ppp_version': PPP_VERSION,
    'fill_gaps': 'false',
    'welfare_type': 'all',
    'format': 'json',
}, timeout=(10, 120))

print('HTTP', r.status_code)
data = r.json()
print(f'records returned: {len(data)}')
print()
if data:
    keep = ['country_name', 'reporting_year', 'reporting_level', 'welfare_type',
            'poverty_line', 'headcount', 'mean', 'gini', 'is_interpolated',
            'estimation_type']
    for k in keep:
        if k in data[0]:
            print(f'  {k:<20} {data[0][k]}')
else:
    raise RuntimeError('Empty response. Check parameter names at '
                       'https://pip.worldbank.org/api before running section 6.')

testing ALB 2017
  diet cost   : 3.04 PPP $/day
  poverty line: 5.85 PPP $/day  (cost / 0.52)

HTTP 200
records returned: 2

  country_name         Albania
  reporting_year       2017
  reporting_level      national
  welfare_type         consumption
  poverty_line         5.85
  headcount            0.1494
  mean                 13.2073
  gini                 0.3306
  is_interpolated      False
  estimation_type      survey


## 5. Query function

In [ ]:
def query_pip(iso3, year, povline):
    """Return PIP records for one country-year at a country-specific line."""
    try:
        r = requests.get(f'{PIP_BASE}/pip', params={
            'country': iso3,
            'year': int(year),
            'povline': float(povline),
            'ppp_version': PPP_VERSION,
            'fill_gaps': 'false',
            'welfare_type': 'all',
            'format': 'json',
        }, timeout=(10, 120))
        r.raise_for_status()
        data = r.json()
        return data if data else None
    except Exception as exc:
        print(f'    {iso3} {year}: {exc}')
        return None

## 6. Run the queries

Cached every 25 requests. Re-executing the cell resumes an interrupted run.

In [ ]:
done, rows = set(), []
if RAW.exists():
    prev = pd.read_csv(RAW)
    rows = prev.to_dict('records')
    done = set(zip(prev['iso3'], prev['year']))
    print(f'resuming: {len(done)} queries already cached')

todo = [r for _, r in grid.iterrows() if (r['iso3'], r['year']) not in done]
print(f'queries to run: {len(todo)}')
print()

for i, r in enumerate(todo, 1):
    res = query_pip(r['iso3'], r['year'], r['povline'])
    if res:
        for rec in res:
            rows.append({
                'iso3': r['iso3'],
                'year': int(r['year']),
                'cohd_ppp': r['cohd_ppp'],
                'povline_used': r['povline'],
                TARGET_NAME: rec.get('headcount'),
                'mean_income': rec.get('mean'),
                'gini_pip': rec.get('gini'),
                'welfare_type': rec.get('welfare_type'),
                'reporting_level': rec.get('reporting_level'),
                'is_interpolated': rec.get('is_interpolated'),
                'estimation_type': rec.get('estimation_type'),
                'ppp_version': rec.get('ppp_version'),
            })
    if i % 25 == 0 or i == len(todo):
        pd.DataFrame(rows).to_csv(RAW, index=False)
        print(f'  {i}/{len(todo)} queried, cached')
    time.sleep(PAUSE)

raw = pd.DataFrame(rows)
raw.to_csv(RAW, index=False)
print()
print(f'raw results: {len(raw)} rows, {raw["iso3"].nunique()} countries')
raw.head()

queries to run: 507

  25/507 queried, cached
  50/507 queried, cached
  75/507 queried, cached
  100/507 queried, cached
  125/507 queried, cached
  150/507 queried, cached
  175/507 queried, cached
  200/507 queried, cached
  225/507 queried, cached
  250/507 queried, cached
  275/507 queried, cached
  300/507 queried, cached
  325/507 queried, cached
  350/507 queried, cached
  375/507 queried, cached
  400/507 queried, cached
  425/507 queried, cached
  450/507 queried, cached
  475/507 queried, cached
  500/507 queried, cached
  507/507 queried, cached

raw results: 540 rows, 129 countries


,iso3,year,cohd_ppp,povline_used,cannot_afford,mean_income,gini_pip,welfare_type,reporting_level,is_interpolated,estimation_type,ppp_version
0,ALB,2017,3.04,5.8462,0.1494,13.2073,0.3306,consumption,national,False,survey,None
1,ALB,2017,3.04,5.8462,0.2706,11.0788,0.3710,income,national,False,survey,None
2,ALB,2018,3.13,6.0192,0.1079,14.1707,0.3015,consumption,national,False,survey,None
3,ALB,2018,3.13,6.0192,0.2521,11.5962,0.3599,income,national,False,survey,None
4,ALB,2019,3.32,6.3846,0.0989,14.6318,0.3012,consumption,national,False,survey,None


In [ ]:
# Verifies the queries used the intended poverty line. The ratio must equal
# 1 / FOOD_BUDGET_SHARE for every row; any other value means stale cached results were
# used and section 6 needs re-running after clearing the cache.
ratio = (raw['povline_used'] / raw['cohd_ppp']).round(3).unique()
expected = round(1 / FOOD_BUDGET_SHARE, 3)

print(f'povline / cohd ratio observed : {ratio}')
print(f'expected                      : [{expected}]')

if len(ratio) == 1 and abs(ratio[0] - expected) < 0.01:
    print('OK -- results reflect the current configuration')
else:
    raise RuntimeError(
        'Poverty line mismatch. The cache was not cleared. Re-run the cache reset '
        'cell in section 1, then re-run section 6.')

povline / cohd ratio observed : [1.923]
expected                      : [1.923]
OK -- results reflect the current configuration


## 7. Collapse to one observation per country-year

In [ ]:
# A single query can return several records: national / urban / rural coverage, and
# income / consumption welfare types. National coverage is preferred, and consumption
# where both welfare types are reported.
raw['_lvl'] = (raw['reporting_level'] == 'national').astype(int)
raw['_wt']  = (raw['welfare_type'] == 'consumption').astype(int)

target = (raw.sort_values(['_lvl', '_wt'], ascending=False)
             .groupby(['iso3', 'year'], as_index=False).first()
             .drop(columns=['_lvl', '_wt']))

# The model is a country-level cross-section, so the most recent survey per country is
# flagged. Earlier surveys are retained for sensitivity checks rather than discarded.
target['is_latest'] = (target.groupby('iso3')['year'].transform('max') == target['year'])

print(f'collapsed: {len(target)} country-years, {target["iso3"].nunique()} countries')
print(f'latest-survey cross-section: {target["is_latest"].sum()} countries')
print()
print(f'{TARGET_NAME} distribution, cross-section:')
print(target.loc[target['is_latest'], TARGET_NAME].describe().round(4).to_string())
print()
print('welfare type mix:')
print(target['welfare_type'].value_counts().to_string())

collapsed: 507 country-years, 129 countries
latest-survey cross-section: 129 countries

cannot_afford distribution, cross-section:
count    129.0000
mean       0.3384
std        0.3389
min        0.0000
25%        0.0209
50%        0.2011
75%        0.6586
max        0.9753

welfare type mix:
welfare_type
income         349
consumption    158


## 8. Comparison with the published series

The published `CoHD_headcount` is imputed from regional aggregates where country data is
missing, so divergence is expected and informative rather than an error. Large gaps
identify countries whose published figure is estimate rather than measurement, and the gap
is carried forward as a column.

High correlation with a modest median gap indicates the units and PPP basis are right. A
near-zero correlation would indicate a genuine configuration problem.

In [ ]:
published = (fpn[fpn['indicator'] == 'CoHD_headcount']
             [['iso3', 'year', 'value']]
             .rename(columns={'value': 'published'}))
if len(published) and published['published'].max() > 1.5:
    published['published'] = published['published'] / 100.0

chk = target.merge(published, on=['iso3', 'year'], how='left')
chk['gap_vs_published'] = chk[TARGET_NAME] - chk['published']

both = chk.dropna(subset=['published'])
corr = both[TARGET_NAME].corr(both['published'])

print(f'country-years with both series : {len(both)}')
print(f'correlation                    : {corr:.4f}')
print(f'median gap                     : {both["gap_vs_published"].median():+.4f}')
print(f'mean absolute gap              : {both["gap_vs_published"].abs().mean():.4f}')
print()
if corr > 0.75:
    print('Correlation is high. Units and PPP basis behave as expected; the remaining')
    print('spread reflects imputation in the published series.')
else:
    print('Correlation is low enough to suspect a configuration problem rather than')
    print('imputation. Re-check PPP_VERSION against the FPN metadata.')

country-years with both series : 507
correlation                    : 0.8558
median gap                     : -0.0357
mean absolute gap              : 0.1063

Correlation is high. Units and PPP basis behave as expected; the remaining
spread reflects imputation in the published series.


In [ ]:
# Where the divergence concentrates. Clustering by region or income group is the
# signature of regional imputation in the published series.
byc = (both.groupby('iso3')['gap_vs_published'].mean().reset_index()
           .merge(coverage[['iso3', 'country', 'region', 'income_group']],
                  on='iso3', how='left'))

print('mean gap by income group:')
print(byc.groupby('income_group')['gap_vs_published']
         .agg(['mean', 'count']).round(3).to_string())
print()
print('mean gap by region:')
print(byc.groupby('region')['gap_vs_published']
         .agg(['mean', 'count']).round(3).to_string())
print()
print('largest divergences (published figure most likely imputed):')
print(byc.reindex(byc['gap_vs_published'].abs().sort_values(ascending=False).index)
         .head(15)[['country', 'region', 'income_group', 'gap_vs_published']]
         .round(3).to_string(index=False))

mean gap by income group:
                      mean  count
income_group                     
High income         -0.122     45
Low income           0.119     17
Lower middle income  0.115     30
Upper middle income -0.038     37

mean gap by region:
                                                    mean  count
region                                                         
East Asia & Pacific                               -0.009     11
Europe & Central Asia                             -0.078     43
Latin America & Caribbean                         -0.093     18
Middle East, North Africa, Afghanistan & Pakistan  0.054     13
North America                                     -0.028      2
South Asia                                         0.158      6
Sub-Saharan Africa                                 0.059     36

largest divergences (published figure most likely imputed):
             country                                            region        income_group  gap_vs_published
   

## 9. Attach controls and save

In [ ]:
vintage = pd.read_csv(CACHE / 'pip_vintage.csv')[['iso3', 'years_stale']]

target = (chk
          .merge(vintage, on='iso3', how='left')
          .merge(coverage[['iso3', 'country', 'region', 'income_group']],
                 on='iso3', how='left'))

target.to_csv(CACHE / 'target.csv', index=False)

cs = target[target['is_latest']]
print(f'saved target.csv: {len(target)} rows, {target["iso3"].nunique()} countries')
print(f'cross-section for modelling (is_latest): {len(cs)} countries')
print()
print(f'{TARGET_NAME} by income group, cross-section:')
print(cs.groupby('income_group')[TARGET_NAME]
        .agg(['count', 'median']).round(3).to_string())
print()
print('region counts (cross-validation groups):')
print(cs['region'].value_counts().to_string())
print()
print(f'countries on surveys older than 10 years: {(cs["years_stale"] > 10).sum()}')
print(f'country-years with no published comparison: {target["published"].isna().sum()}')
target.head()

saved target.csv: 507 rows, 129 countries
cross-section for modelling (is_latest): 129 countries

cannot_afford by income group, cross-section:
                     count  median
income_group                      
High income             45   0.011
Low income              17   0.910
Lower middle income     30   0.646
Upper middle income     37   0.217

region counts (cross-validation groups):
region
Europe & Central Asia                                43
Sub-Saharan Africa                                   36
Latin America & Caribbean                            18
Middle East, North Africa, Afghanistan & Pakistan    13
East Asia & Pacific                                  11
South Asia                                            6
North America                                         2

countries on surveys older than 10 years: 0
country-years with no published comparison: 0


,iso3,year,cohd_ppp,povline_used,cannot_afford,mean_income,gini_pip,welfare_type,reporting_level,is_interpolated,estimation_type,ppp_version,is_latest,published,gap_vs_published,years_stale,country,region,income_group
0,AGO,2018,3.27,6.2885,0.7025,6.2181,0.5126,consumption,national,False,survey,None,True,0.648,0.0545,8,Angola,Sub-Saharan Africa,Lower middle income
1,ALB,2017,3.04,5.8462,0.1494,13.2073,0.3306,consumption,national,False,survey,None,False,0.292,-0.1426,6,Albania,Europe & Central Asia,Upper middle income
2,ALB,2018,3.13,6.0192,0.1079,14.1707,0.3015,consumption,national,False,survey,None,False,0.209,-0.1011,6,Albania,Europe & Central Asia,Upper middle income
3,ALB,2019,3.32,6.3846,0.0989,14.6318,0.3012,consumption,national,False,survey,None,False,0.184,-0.0851,6,Albania,Europe & Central Asia,Upper middle income
4,ALB,2020,3.40,6.5385,0.1007,14.8472,0.2942,consumption,national,False,survey,None,True,0.176,-0.0753,6,Albania,Europe & Central Asia,Upper middle income


## 10. Persist

`target.csv` costs several hundred API calls to rebuild, so it is written back to Drive
before the session ends. The modelling notebook reads it from there.

In [ ]:
!zip -qr data_cache.zip data_cache
!cp data_cache.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/data_cache.zip
print()
print('data_cache.zip updated in Drive, now containing target.csv')

-rw------- 1 root root 427K Sep 10 14:24 /content/drive/MyDrive/data_cache.zip

data_cache.zip updated in Drive, now containing target.csv
